# 05 Review outputs

Review the latest inventory, rule-classification, and text-extraction outputs together before any rename planning or execution.


In [2]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

OUTPUTS_DIR = PROJECT_ROOT / 'data' / 'outputs'
EXPORT_REVIEW_SNAPSHOT = True


In [3]:
from src.reporting import detect_latest_outputs, load_optional_parquet, build_review_frame, review_summary
from src.inventory import ensure_inventory_schema

paths = detect_latest_outputs(OUTPUTS_DIR)
print(paths)

inv = load_optional_parquet(paths.inventory_path)
classified = load_optional_parquet(paths.classification_path)
text_df = load_optional_parquet(paths.text_path)

if inv is not None:
    inv = ensure_inventory_schema(inv)

review = build_review_frame(inv, classified, text_df)
print('Rows in review frame:', len(review))


ReviewPaths(inventory_path=WindowsPath('c:/00_dev/SCH-FILE-ORGANIZER/data/outputs/inventory_HTL0049-01_OITYLO-KOKKALA_MANI_20260309_133149.parquet'), classification_path=None, text_path=WindowsPath('c:/00_dev/SCH-FILE-ORGANIZER/data/outputs/inventory_with_text_20260309_154442.parquet'))
Rows in review frame: 4241


In [4]:
summary = review_summary(review)
pd.DataFrame([summary])


,rows,duplicates,junk_candidates,manual_review,text_errors,long_paths
0,4241,2660,0,0,1,1375


In [5]:
display(review[['relative_path', 'suffix', 'size_bytes', 'rule_status', 'text_status', 'path_length']].head(20))
display(review['rule_status'].value_counts(dropna=False).rename_axis('rule_status').reset_index(name='count'))
display(review['text_status'].value_counts(dropna=False).rename_axis('text_status').reset_index(name='count'))
display(review['suffix'].fillna('').value_counts(dropna=False).rename_axis('suffix').reset_index(name='count').head(20))


,relative_path,suffix,size_bytes,rule_status,text_status,path_length
0,.DS_Store,,14340,not_classified,unsupported,63
1,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\.DS_Store,,10244,not_classified,unsupported,88
2,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,152816,not_classified,ok,145
3,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,124890,not_classified,ok,156
4,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΔΗΛΩΣΕΙΣ ΑΝΑΘΕΣΗΣ - Α...,.pdf,67290,not_classified,empty,159
5,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΓΚΡΙΣΗ ΕΦΟΡΕΙΑΣ ΑΡΧΑ...,.pdf,564756,not_classified,ok,146
6,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΕΛΕΓΧΟΣ ΔΟΜΗΣΗΣ\ΠΟΡΙΣ...,.pdf,415711,not_classified,ok,145
7,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\01_SONADO IK...,.pdf,676187,not_classified,ok,124
8,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\02_SONADO IK...,.pdf,2386336,not_classified,ok,130
9,ΑΔΕΙΟΔΟΤΗΣΕΙΣ - ΚΑΤΟΨΕΙΣ\ΚΑΤΟΨΕΙΣ\03_SONADO IK...,.pdf,12993826,not_classified,ok,126


,rule_status,count
0,not_classified,4241


,text_status,count
0,ok,1926
1,unsupported,1677
2,empty,637
3,error,1


,suffix,count
0,.pdf,2403
1,.jpeg,664
2,.jpg,355
3,.dwg,253
4,.msg,121
5,.bak,78
6,.docx,68
7,,64
8,.db,50
9,.xlsx,49


In [6]:
junk = review[review['rule_status'] == 'archive_or_delete_candidate'].copy()
duplicates = review[review.get('is_duplicate_hash', False).fillna(False)].copy()
manual_review = review[review['needs_manual_review']].copy()
text_errors = review[review['has_text_error']].copy()
long_paths = review[review['long_path_warning']].copy()
compliant = review[review['rule_status'] == 'compliant_keep_review_path'].copy()

display(junk[['relative_path', 'rule_reason', 'proposed_relative_target']].head(20))
display(duplicates[['relative_path', 'hash', 'duplicate_group_size', 'rule_status']].head(20))
display(text_errors[['relative_path', 'suffix', 'text_source', 'text_error']].head(20))
display(long_paths[['relative_path', 'path_length', 'filename_length', 'rule_status']].sort_values('path_length', ascending=False).head(20))
display(compliant[['relative_path', 'parsed_phase', 'parsed_doc_type', 'proposed_relative_target']].head(20))
display(manual_review[['relative_path', 'suffix', 'rule_reason', 'text_status', 'text_preview']].head(30))


KeyError: "['proposed_relative_target'] not in index"

In [7]:
review_candidates = review[review['needs_manual_review'] | review['has_text_error'] | review['long_path_warning']].copy()
review_candidates = review_candidates.sort_values(['needs_manual_review', 'has_text_error', 'path_length', 'relative_path'], ascending=[False, False, False, True])
display(review_candidates[['relative_path', 'suffix', 'rule_status', 'rule_reason', 'text_status', 'path_length', 'text_preview']].head(50))


,relative_path,suffix,rule_status,rule_reason,text_status,path_length,text_preview
4054,ΦΑΚΕΛΟΣ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ\ΕΓΚΡΙΣΕΙΣ - ΑΠΟΦΑΣΕ...,.pdf,not_classified,classification output not loaded,error,165,
4168,ΦΑΚΕΛΟΣ ΑΝΑΠΤΥΞΙΑΚΟΥ ΝΟΜΟΥ\ΛΙΣΤΑ ΔΙΚΑΙΟΛΟΓΗΤΙΚ...,.pdf,not_classified,classification output not loaded,empty,311,
3766,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,308,ΕΓΚΥΡΟ ΑΝΤΙΓΡΑΦΟ Α/Α Πράξης: 383786 71B5B41946...
3767,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,308,ΕΓΚΥΡΟ ΑΝΤΙΓΡΑΦΟ Α/Α Πράξης: 383786 97BF5C923C...
3768,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,308,ΕΓΚΥΡΟ ΑΝΤΙΓΡΑΦΟ Α/Α Πράξης: 383786 4F2F0AABA6...
3769,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,308,ΕΓΚΥΡΟ ΑΝΤΙΓΡΑΦΟ Α/Α Πράξης: 383786 00AF7C8967...
3750,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,306,ΜΕΛΕΤΗ ΑΣΘΕΝΩΝ ΡΕΥΜΑΤΩΝ ΤΕΧΝΙΚΗ ΠΕΡΙΓΡΑΦΗ Εργο...
3756,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,305,ΜΕΛΕΤΗ HΛΕΚΤΡΟΛΟΓΙΚΗΣ ΕΓΚΑΤΑΣΤΑΣΗΣ ΤΕΧΝΙΚΗ ΠΕΡ...
3770,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,305,ΕΓΚΥΡΟ ΑΝΤΙΓΡΑΦΟ Α/Α Πράξης: 383786 45D04A842E...
3825,ΤΕΧΝΙΚΟΣ ΦΑΚΕΛΟΣ\ΟΙΚΟΔΟΜΙΚΕΣ_ΑΔΕΙΕΣ\ΑΔΕΙΕΣ ΜΕ ...,.pdf,not_classified,classification output not loaded,ok,304,Μ Ε Λ Ε Τ Η Ε Ν Ε Ρ Γ Η Τ Ι Κ Η Σ Π Υ Ρ Ο Π Ρ ...


In [8]:
if EXPORT_REVIEW_SNAPSHOT:
    OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
    review_path = OUTPUTS_DIR / 'review_snapshot_latest.parquet'
    review_csv = OUTPUTS_DIR / 'review_snapshot_latest.csv'
    review.to_parquet(review_path, index=False)
    review.to_csv(review_csv, index=False, encoding='utf-8-sig')
    print('Saved:', review_path)
    print('Saved:', review_csv)


Saved: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\review_snapshot_latest.parquet
Saved: c:\00_dev\SCH-FILE-ORGANIZER\data\outputs\review_snapshot_latest.csv
